# Tempreture Persona Transformer Training

This notebook trains a PyTorch Transformer classifier for temperature/HVAC persona prediction using `temperatureData.csv`. It mounts Google Drive, builds recent room temperature sequences, trains on the `ac_persona` label, and saves model artifacts back to Drive as a zip file.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATE_DATA_PATHS = [
    DRIVE_ROOT / 'VisualizationApp' / 'temperatureData.csv',
    DRIVE_ROOT / 'Data' / 'temperatureData.csv',
    DRIVE_ROOT / 'temperatureData.csv',
]

DATA_CSV = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), CANDIDATE_DATA_PATHS[0])
OUT_DIR = DRIVE_ROOT / 'VisualizationApp' / 'AIModelsAndAlgorithms' / 'TempreturePersona' / 'transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_CSV:', DATA_CSV)
print('DATA_CSV exists:', DATA_CSV.exists())
print('OUT_DIR:', OUT_DIR)

if not DATA_CSV.exists():
    raise FileNotFoundError(
        'Could not find temperatureData.csv in the default Drive locations. '
        'Set DATA_CSV manually to your file path, then rerun this cell.'
    )


DATA_CSV: /content/drive/MyDrive/VisualizationApp/temperatureData.csv
DATA_CSV exists: True
OUT_DIR: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer


In [3]:
import json
import math
import random
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


device: cuda


In [4]:
# Training controls. Start smaller for a smoke test, then increase for full training.
LABEL_COL = 'ac_persona'
MAX_ROWS = None          # Example quick run: 1_000_000. Use None for all rows.
MAX_SEQUENCES = 500_000  # Increase later if Colab RAM allows.
SEQ_LEN = 24             # 24 samples = 2 hours if data is every 5 minutes.
STRIDE = 3               # Use every 3rd possible sequence to reduce training size.
BATCH_SIZE = 512
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 96
N_HEADS = 4
N_LAYERS = 2
DIM_FEEDFORWARD = 192
DROPOUT = 0.15
PATIENCE = 4

# Most datasets use None for no meaningful AC persona. Keep this default if you want the model
# to learn only real temperature persona classes. Set to an empty set to include every label.
EXCLUDED_LABELS = {'None', 'Unknown', 'nan', ''}


In [5]:
USECOLS = [
    'timestamp', 'room_number', 'floor', 'facade', 'room_type', 'size_m2',
    'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'hvac_mode',
    'ac_persona', 'occupant_state', 'pir_persona', 'room_state', 'pir_motion', 'guest_id',
]

BASE_NUMERIC_COLS = [
    'floor_scaled', 'size_scaled', 'outside_temp_scaled', 'room_temp_scaled',
    'setpoint_scaled', 'ideal_temp_scaled', 'temp_error_scaled', 'comfort_error_scaled',
    'pir_motion', 'has_guest', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
]
CAT_COLS = ['facade', 'room_type', 'hvac_mode', 'occupant_state', 'pir_persona', 'room_state']

def cyclic(values, period):
    radians = 2 * np.pi * values / period
    return np.sin(radians), np.cos(radians)

def scale_temp(series):
    return ((pd.to_numeric(series, errors='coerce').fillna(0).clip(-20, 50) + 20) / 70).astype('float32')

def load_and_prepare(data_csv, max_rows=None):
    df = pd.read_csv(data_csv, usecols=USECOLS, parse_dates=['timestamp'], nrows=max_rows)
    if LABEL_COL not in df.columns:
        raise ValueError(f'LABEL_COL={LABEL_COL!r} was not found. Available columns: {list(df.columns)}')

    df = df.dropna(subset=['timestamp']).copy()
    df = df.sort_values(['room_number', 'timestamp']).reset_index(drop=True)

    df[LABEL_COL] = df[LABEL_COL].fillna('Unknown').astype(str)
    if EXCLUDED_LABELS:
        df = df[~df[LABEL_COL].isin(EXCLUDED_LABELS)].copy()
    if df.empty:
        raise ValueError('No labeled rows left after filtering EXCLUDED_LABELS. Adjust EXCLUDED_LABELS.')
    df = df.sort_values(['room_number', 'timestamp']).reset_index(drop=True)

    for col in ['floor', 'size_m2', 'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'pir_motion']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['guest_id'] = pd.to_numeric(df['guest_id'], errors='coerce')

    for col in CAT_COLS:
        df[col] = df[col].fillna('Unknown').astype(str)

    df['floor_scaled'] = df['floor'].clip(0, 30) / 30.0
    df['size_scaled'] = df['size_m2'].clip(0, 120) / 120.0
    df['outside_temp_scaled'] = scale_temp(df['outside_temp'])
    df['room_temp_scaled'] = scale_temp(df['room_temp'])
    df['setpoint_scaled'] = scale_temp(df['setpoint'])
    df['ideal_temp_scaled'] = scale_temp(df['ideal_temp'])
    df['temp_error_scaled'] = ((df['room_temp'] - df['setpoint']).clip(-20, 20) + 20) / 40.0
    df['comfort_error_scaled'] = ((df['room_temp'] - df['ideal_temp']).clip(-20, 20) + 20) / 40.0
    df['pir_motion'] = df['pir_motion'].clip(0, 1).astype('float32')
    df['has_guest'] = df['guest_id'].notna().astype('float32')
    df['hour_sin'], df['hour_cos'] = cyclic(df['timestamp'].dt.hour, 24)
    df['dow_sin'], df['dow_cos'] = cyclic(df['timestamp'].dt.dayofweek, 7)

    for col in BASE_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('float32')
    return df

df = load_and_prepare(DATA_CSV, MAX_ROWS)
print('rows:', len(df))
print('time range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print('labels:')
print(df[LABEL_COL].value_counts())
df.head()


rows: 2225132
time range: 2022-01-06 19:05:00 to 2022-07-06 00:00:00
labels:
ac_persona
Reactive           831752
AlwaysOnComfort    691292
EnergySaver        674206
Preconditioning     18261
Housekeeping         9621
Name: count, dtype: int64


,timestamp,room_number,floor,facade,room_type,size_m2,outside_temp,room_temp,setpoint,ideal_temp,...,room_temp_scaled,setpoint_scaled,ideal_temp_scaled,temp_error_scaled,comfort_error_scaled,has_guest,hour_sin,hour_cos,dow_sin,dow_cos
0,2022-01-22 07:20:00,1,1,East,Deluxe,35,11.58,14.93,20.0,22.0,...,0.499000,0.571429,0.6,0.37325,0.32325,0.0,0.965926,-0.258819,-0.974928,-0.222521
1,2022-01-22 07:25:00,1,1,East,Deluxe,35,11.60,15.25,20.0,22.0,...,0.503571,0.571429,0.6,0.38125,0.33125,0.0,0.965926,-0.258819,-0.974928,-0.222521
2,2022-01-22 07:30:00,1,1,East,Deluxe,35,11.62,15.58,20.0,22.0,...,0.508286,0.571429,0.6,0.38950,0.33950,0.0,0.965926,-0.258819,-0.974928,-0.222521
3,2022-01-22 07:35:00,1,1,East,Deluxe,35,11.64,15.91,20.0,22.0,...,0.513000,0.571429,0.6,0.39775,0.34775,0.0,0.965926,-0.258819,-0.974928,-0.222521
4,2022-01-22 07:40:00,1,1,East,Deluxe,35,11.66,16.23,20.0,22.0,...,0.517571,0.571429,0.6,0.40575,0.35575,0.0,0.965926,-0.258819,-0.974928,-0.222521


In [6]:
classes = sorted(df[LABEL_COL].unique().tolist())
class_to_id = {label: i for i, label in enumerate(classes)}
df['label_id'] = df[LABEL_COL].map(class_to_id).astype('int64')

category_values = {}
for col in CAT_COLS:
    values = sorted(df[col].fillna('Unknown').astype(str).unique().tolist())
    if 'Unknown' not in values:
        values.append('Unknown')
    category_values[col] = values

feature_names = list(BASE_NUMERIC_COLS)
for col, values in category_values.items():
    feature_names.extend([f'{col}={value}' for value in values])

def encode_features(frame):
    parts = [frame[BASE_NUMERIC_COLS].to_numpy(dtype='float32')]
    for col, values in category_values.items():
        mapping = {value: i for i, value in enumerate(values)}
        unknown_idx = mapping.get('Unknown', 0)
        ids = frame[col].fillna('Unknown').astype(str).map(mapping).fillna(unknown_idx).astype(int).to_numpy()
        onehot = np.zeros((len(frame), len(values)), dtype='float32')
        onehot[np.arange(len(frame)), ids] = 1.0
        parts.append(onehot)
    return np.concatenate(parts, axis=1)

encoded = encode_features(df)
labels = df['label_id'].to_numpy(dtype='int64')
print('classes:', classes)
print('feature count:', encoded.shape[1])
print('categories:', {k: len(v) for k, v in category_values.items()})


classes: ['AlwaysOnComfort', 'EnergySaver', 'Housekeeping', 'Preconditioning', 'Reactive']
feature count: 42
categories: {'facade': 5, 'room_type': 4, 'hvac_mode': 4, 'occupant_state': 6, 'pir_persona': 5, 'room_state': 4}


In [7]:
def build_sequences(frame, encoded_features, labels, seq_len, stride=1, max_sequences=None):
    sequences = []
    y = []
    end_indices = []

    frame = frame.reset_index(drop=True)
    for _, group in frame.groupby('room_number', sort=False):
        idx = group.index.to_numpy(dtype=np.int64)
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx), stride):
            window_idx = idx[end - seq_len + 1:end + 1]
            sequences.append(encoded_features[window_idx])
            y.append(labels[idx[end]])
            end_indices.append(idx[end])
            if max_sequences and len(sequences) >= max_sequences:
                return (
                    np.stack(sequences).astype('float32'),
                    np.array(y, dtype='int64'),
                    np.array(end_indices, dtype=np.int64),
                )

    if not sequences:
        raise ValueError('No sequences were built. Reduce SEQ_LEN or check labeled row counts.')
    return (
        np.stack(sequences).astype('float32'),
        np.array(y, dtype='int64'),
        np.array(end_indices, dtype=np.int64),
    )

x, y, end_indices = build_sequences(df, encoded, labels, SEQ_LEN, STRIDE, MAX_SEQUENCES)
print('x:', x.shape, 'y:', y.shape)


x: (500000, 24, 42) y: (500000,)


In [8]:
# Chronological split by sequence end timestamp.
sequence_times = df.loc[end_indices, 'timestamp'].reset_index(drop=True)
order = np.argsort(sequence_times.to_numpy())
x = x[order]
y = y[order]
end_indices = end_indices[order]

n = len(x)
train_end = int(n * 0.80)
val_end = int(n * 0.90)
x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:val_end], y[train_end:val_end]
x_test, y_test = x[val_end:], y[val_end:]
test_indices = end_indices[val_end:]

print('train:', x_train.shape, 'val:', x_val.shape, 'test:', x_test.shape)
print('split times:', sequence_times.iloc[train_end], sequence_times.iloc[val_end])


train: (400000, 24, 42) val: (50000, 24, 42) test: (50000, 24, 42)
split times: 2022-05-04 04:10:00 2022-04-01 10:15:00


In [9]:
class_counts = np.bincount(y_train, minlength=len(classes)).astype('float32')
class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = class_weights / class_weights.mean()
print('class_counts:', dict(zip(classes, class_counts.astype(int).tolist())))
print('class_weights:', dict(zip(classes, class_weights.round(3).tolist())))

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)


class_counts: {'AlwaysOnComfort': 127236, 'EnergySaver': 124145, 'Housekeeping': 2085, 'Preconditioning': 3703, 'Reactive': 142831}
class_weights: {'AlwaysOnComfort': 0.050999999046325684, 'EnergySaver': 0.052000001072883606, 'Housekeeping': 3.1040000915527344, 'Preconditioning': 1.7480000257492065, 'Reactive': 0.04500000178813934}


In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TempreturePersonaTransformer(nn.Module):
    def __init__(self, input_dim, n_classes, d_model, n_heads, n_layers, dim_feedforward, dropout):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.position = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        hidden = self.input_projection(x)
        hidden = self.position(hidden)
        hidden = self.encoder(hidden)
        pooled = hidden.mean(dim=1)
        return self.head(pooled)

model = TempreturePersonaTransformer(
    input_dim=x_train.shape[-1],
    n_classes=len(classes),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
print(model)


/tmp/ipykernel_5182/992374405.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


TempreturePersonaTransformer(
  (input_projection): Linear(in_features=42, out_features=96, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
        )
        (linear1): Linear(in_features=96, out_features=192, bias=True)
        (dropout): Dropout(p=0.15, inplace=False)
        (linear2): Linear(in_features=192, out_features=96, bias=True)
        (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.15, inplace=False)
        (dropout2): Dropout(p=0.15, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=96, out_features=96, bias=True)
    (2): GELU(ap

In [11]:
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * len(xb)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_count += len(xb)
    return total_loss / max(total_count, 1), total_correct / max(total_count, 1)

best_val = float('inf')
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_acc, 'val_loss': val_loss, 'val_accuracy': val_acc})
    print(f'epoch {epoch:02d} train_loss={train_loss:.5f} train_acc={train_acc:.4f} val_loss={val_loss:.5f} val_acc={val_acc:.4f}')

    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('early stopping')
            break

if best_state is not None:
    model.load_state_dict(best_state)


epoch 01 train_loss=0.59246 train_acc=0.5824 val_loss=0.71822 val_acc=0.6373
epoch 02 train_loss=0.31817 train_acc=0.7717 val_loss=0.45654 val_acc=0.7710
epoch 03 train_loss=0.20909 train_acc=0.8612 val_loss=0.34245 val_acc=0.8688
epoch 04 train_loss=0.15801 train_acc=0.8951 val_loss=0.28580 val_acc=0.8793
epoch 05 train_loss=0.13420 train_acc=0.9115 val_loss=0.48777 val_acc=0.8423
epoch 06 train_loss=0.11826 train_acc=0.9215 val_loss=0.36997 val_acc=0.8783
epoch 07 train_loss=0.10742 train_acc=0.9282 val_loss=0.38655 val_acc=0.8651
epoch 08 train_loss=0.08652 train_acc=0.9413 val_loss=0.39392 val_acc=0.8909
early stopping


In [12]:
@torch.no_grad()
def predict(loader):
    model.eval()
    probs = []
    actual = []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        probs.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
        actual.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(actual)

probs, actual = predict(test_loader)
pred = probs.argmax(axis=1)
accuracy = float((pred == actual).mean())
report = classification_report(actual, pred, target_names=classes, digits=4, zero_division=0)
cm = confusion_matrix(actual, pred, labels=list(range(len(classes))))
print('accuracy:', accuracy)
print(report)


accuracy: 0.8055
                 precision    recall  f1-score   support

AlwaysOnComfort     0.9669    0.8428    0.9006     15641
    EnergySaver     0.6568    0.8928    0.7568     13163
   Housekeeping     1.0000    1.0000    1.0000        44
Preconditioning     1.0000    0.9895    0.9947        95
       Reactive     0.8291    0.7220    0.7719     21057

       accuracy                         0.8055     50000
      macro avg     0.8906    0.8894    0.8848     50000
   weighted avg     0.8273    0.8055    0.8088     50000



In [13]:
model_path = OUT_DIR / 'tempreture_persona_transformer.pt'
metadata_path = OUT_DIR / 'tempreture_persona_transformer_metadata.json'
report_path = OUT_DIR / 'tempreture_persona_transformer_report.txt'
cm_path = OUT_DIR / 'tempreture_persona_transformer_confusion_matrix.csv'
zip_path = OUT_DIR / 'tempreture_persona_transformer_outputs.zip'

metadata = {
    'model_type': 'transformer_classifier',
    'task': 'tempreture_persona_prediction',
    'label_col': LABEL_COL,
    'classes': classes,
    'class_to_id': class_to_id,
    'sequence_length': SEQ_LEN,
    'stride': STRIDE,
    'feature_names': feature_names,
    'base_numeric_cols': BASE_NUMERIC_COLS,
    'categorical_cols': CAT_COLS,
    'category_values': category_values,
    'input_dim': int(x_train.shape[-1]),
    'd_model': D_MODEL,
    'n_heads': N_HEADS,
    'n_layers': N_LAYERS,
    'dim_feedforward': DIM_FEEDFORWARD,
    'dropout': DROPOUT,
    'seed': SEED,
    'max_rows': MAX_ROWS,
    'max_sequences': MAX_SEQUENCES,
    'excluded_labels': sorted(EXCLUDED_LABELS),
    'history': history,
    'metrics': {'accuracy': accuracy},
}

torch.save({'state_dict': model.state_dict(), 'metadata': metadata}, model_path)
metadata_path.write_text(json.dumps(metadata, indent=2))

cm_df = pd.DataFrame(cm, index=classes, columns=classes)
cm_df.to_csv(cm_path)

full_report = '\n'.join([
    f'rows: {len(df):,}',
    f'sequences: {len(x):,}',
    f'train sequences: {len(x_train):,}',
    f'validation sequences: {len(x_val):,}',
    f'test sequences: {len(x_test):,}',
    f'sequence length: {SEQ_LEN}',
    f'accuracy: {accuracy:.4f}',
    '',
    report,
])
report_path.write_text(full_report)
print(full_report)


rows: 2,225,132
sequences: 500,000
train sequences: 400,000
validation sequences: 50,000
test sequences: 50,000
sequence length: 24
accuracy: 0.8055

                 precision    recall  f1-score   support

AlwaysOnComfort     0.9669    0.8428    0.9006     15641
    EnergySaver     0.6568    0.8928    0.7568     13163
   Housekeeping     1.0000    1.0000    1.0000        44
Preconditioning     1.0000    0.9895    0.9947        95
       Reactive     0.8291    0.7220    0.7719     21057

       accuracy                         0.8055     50000
      macro avg     0.8906    0.8894    0.8848     50000
   weighted avg     0.8273    0.8055    0.8088     50000



In [14]:
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_path, metadata_path, report_path, cm_path]:
        zf.write(path, arcname=path.name)

print('saved model:', model_path)
print('saved metadata:', metadata_path)
print('saved report:', report_path)
print('saved confusion matrix:', cm_path)
print('created zip:', zip_path)
print('You can download the zip from Google Drive:', zip_path)


saved model: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer.pt
saved metadata: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_metadata.json
saved report: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_report.txt
saved confusion matrix: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_confusion_matrix.csv
created zip: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_outputs.zip
You can download the zip from Google Drive: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_outputs.zip
